# Stanford RNA 3D Folding Part 2 - Competition Submission

This notebook predicts 3D structures of RNA molecules from their sequences.

## Competition Overview
- **Goal**: Predict 3D RNA structures using only sequence information
- **Evaluation**: TM-score (0.0 to 1.0, higher is better)
- **Output**: 5 structure predictions per sequence with C1' atom coordinates

In [15]:
# Install/upgrade required packages (for Kaggle environment or if missing locally)
# This will install packages in the current Python environment
import subprocess
import sys

def install_package(package):
    """Install a package, but don't fail if network is unavailable."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package], 
                             timeout=60, stderr=subprocess.DEVNULL)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        raise Exception(f"Failed to install {package}: {e}")

try:
    import numpy
except ImportError:
    try:
        print("Installing numpy...")
        install_package("numpy>=1.24.0")
    except Exception as e:
        print(f"Error: Could not install numpy: {e}")
        raise  # numpy is required, so fail if we can't install it

try:
    import pandas
except ImportError:
    try:
        print("Installing pandas...")
        install_package("pandas>=2.0.0")
    except Exception as e:
        print(f"Error: Could not install pandas: {e}")
        raise  # pandas is required, so fail if we can't install it

try:
    import scipy
except ImportError:
    try:
        print("Installing scipy...")
        install_package("scipy>=1.10.0")
    except Exception as e:
        print(f"Warning: Could not install scipy: {e}")
        print("Continuing without scipy (may not be needed for basic predictions)...")
        scipy = None

try:
    import Bio
except ImportError:
    try:
        print("Installing biopython...")
        install_package("biopython>=1.81")
    except Exception as e:
        print(f"Warning: Could not install biopython: {e}")
        print("Continuing without biopython (not required for basic predictions)...")
        Bio = None

try:
    import torch
except ImportError:
    try:
        print("Installing torch...")
        install_package("torch>=2.0.0")
    except Exception as e:
        print(f"Warning: Could not install torch: {e}")
        print("Continuing without torch (not required for basic predictions)...")
        torch = None

print("✓ Package installation check complete!")

# Import required libraries
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path

# Add utils to path (works for both local and Kaggle)
if os.path.exists('/kaggle/working'):
    sys.path.append('/kaggle/working')
else:
    # For local development, add current directory
    sys.path.append(os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.')))

try:
    from utils import (
        read_test_sequences,
        generate_submission_template,
        save_submission,
        validate_submission,
        parse_fasta,
        read_msa_file,
        clip_coordinates
    )
    print("✓ Utils imported successfully!")
except ImportError as e:
    print(f"Warning: Could not import utils: {e}")
    print("Make sure utils.py is in the same directory or /kaggle/working")
    print("Using fallback implementations...")
    
    # Fallback implementations if utils module not available
    def read_test_sequences(file_path: str = "test_sequences.csv") -> pd.DataFrame:
        """Fallback: Read test sequences CSV file."""
        # If file_path is already absolute, use it directly
        if os.path.isabs(file_path):
            possible_paths = [file_path]
        else:
            possible_paths = [
                file_path,
                f"/kaggle/input/stanford-rna-3d-folding-2/{os.path.basename(file_path)}",
                os.path.join("/kaggle/working", file_path),
            ]
        # Also try the absolute path even if relative was provided
        if not os.path.isabs(file_path):
            possible_paths.append(f"/kaggle/input/stanford-rna-3d-folding-2/{file_path}")
        
        for path in possible_paths:
            if os.path.exists(path):
                return pd.read_csv(path)
        raise FileNotFoundError(f"Could not find test_sequences.csv. Tried: {possible_paths}")
    
    def generate_submission_template(sequences_df: pd.DataFrame) -> pd.DataFrame:
        """Fallback: Generate submission template DataFrame."""
        submission_rows = []
        for _, row in sequences_df.iterrows():
            target_id = row['target_id']
            sequence = row['sequence']
            for i, residue in enumerate(sequence, start=1):
                resname = residue.upper()
                submission_id = f"{target_id}_{i}"
                submission_row = {
                    'ID': submission_id,
                    'resname': resname,
                    'resid': i,
                    'x_1': 0.0, 'y_1': 0.0, 'z_1': 0.0,
                    'x_2': 0.0, 'y_2': 0.0, 'z_2': 0.0,
                    'x_3': 0.0, 'y_3': 0.0, 'z_3': 0.0,
                    'x_4': 0.0, 'y_4': 0.0, 'z_4': 0.0,
                    'x_5': 0.0, 'y_5': 0.0, 'z_5': 0.0,
                }
                submission_rows.append(submission_row)
        return pd.DataFrame(submission_rows)
    
    def clip_coordinates(coords: np.ndarray) -> np.ndarray:
        """Fallback: Clip coordinates to valid range."""
        return np.clip(coords, -999.999, 9999.999)
    
    def save_submission(predictions_df: pd.DataFrame, output_path: str = "submission.csv"):
        """Fallback: Save submission CSV file."""
        # Clip coordinates to valid range before saving
        coord_cols = [col for col in predictions_df.columns if col.startswith(('x_', 'y_', 'z_'))]
        for col in coord_cols:
            predictions_df[col] = clip_coordinates(predictions_df[col].values)
        
        # Create directory if it doesn't exist (for local development)
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir, exist_ok=True)
            print(f"Created directory: {output_dir}")
        
        predictions_df.to_csv(output_path, index=False)
        print(f"Submission saved to {output_path}")
        print(f"Shape: {predictions_df.shape}")
        print(f"Columns: {predictions_df.columns.tolist()}")
    
    def validate_submission(submission_df: pd.DataFrame, sequences_df: pd.DataFrame) -> bool:
        """Fallback: Validate submission format."""
        required_cols = ['ID', 'resname', 'resid', 
                        'x_1', 'y_1', 'z_1', 'x_2', 'y_2', 'z_2',
                        'x_3', 'y_3', 'z_3', 'x_4', 'y_4', 'z_4',
                        'x_5', 'y_5', 'z_5']
        missing_cols = [col for col in required_cols if col not in submission_df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        print("Submission validation passed!")
        return True
    
    def parse_fasta(fasta_string: str) -> dict:
        """Fallback: Parse FASTA string."""
        import re
        chains = {}
        current_header = None
        current_sequence = []
        for line in fasta_string.split('\n'):
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header and current_sequence:
                    chain_match = re.search(r'chain=([A-Za-z0-9]+)', current_header)
                    chain_id = chain_match.group(1) if chain_match else current_header.split()[0] if current_header else 'A'
                    chains[chain_id] = ''.join(current_sequence)
                current_header = line[1:]
                current_sequence = []
            else:
                current_sequence.append(line)
        if current_header and current_sequence:
            chain_match = re.search(r'chain=([A-Za-z0-9]+)', current_header)
            chain_id = chain_match.group(1) if chain_match else current_header.split()[0] if current_header else 'A'
            chains[chain_id] = ''.join(current_sequence)
        return chains
    
    def read_msa_file(target_id: str, msa_dir: str = "MSA"):
        """Fallback: Read MSA file."""
        possible_paths = [
            os.path.join(msa_dir, f"{target_id}.MSA.fasta"),
            os.path.join("/kaggle/input/stanford-rna-3d-folding-2", msa_dir, f"{target_id}.MSA.fasta"),
            os.path.join("/kaggle/working", msa_dir, f"{target_id}.MSA.fasta"),
        ]
        for msa_path in possible_paths:
            if os.path.exists(msa_path):
                alignments = []
                current_header = None
                current_sequence = []
                with open(msa_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        if line.startswith('>'):
                            if current_header and current_sequence:
                                alignments.append((current_header, ''.join(current_sequence)))
                            current_header = line[1:]
                            current_sequence = []
                        else:
                            current_sequence.append(line)
                    if current_header and current_sequence:
                        alignments.append((current_header, ''.join(current_sequence)))
                return alignments
        return None

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Working directory: {os.getcwd()}")
print("Libraries imported successfully!")

✓ Package installation check complete!
✓ Utils imported successfully!
Python version: 3.9.6 (default, Dec  2 2025, 07:27:58) 
[Clang 17.0.0 (clang-1700.6.3.2)]
Python executable: /Library/Developer/CommandLineTools/usr/bin/python3
Working directory: /Users/nickmoore/kagglecomp
Libraries imported successfully!


## Load Test Sequences

In [16]:
# Read test sequences
INPUT_FILE = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"

try:
    test_sequences = read_test_sequences(INPUT_FILE)
    print(f"Loaded {len(test_sequences)} test sequences")
    print(f"\nColumns: {test_sequences.columns.tolist()}")
    print(f"\nFirst few sequences:")
    print(test_sequences[['target_id', 'sequence']].head())
    print(f"\nSequence lengths: {test_sequences['sequence'].str.len().describe()}")
    print(f"\nSample target_id: {test_sequences['target_id'].iloc[0]}")
except (FileNotFoundError, NameError) as e:
    # For local testing, create a sample file structure
    print(f"Test sequences file not found or read_test_sequences not available: {e}")
    print("Creating sample structure for development...")
    test_sequences = pd.DataFrame({
        'target_id': ['1ABC_A', '2DEF_B'],
        'sequence': ['GGCGUAGUCC', 'AUCGAUCGAU'],
        'temporal_cutoff': ['2025-01-01', '2025-01-02'],
        'description': ['Sample RNA 1', 'Sample RNA 2'],
        'stoichiometry': ['A:1', 'B:1'],
        'all_sequences': ['>Chain A\nGGCGUAGUCC', '>Chain B\nAUCGAUCGAU'],
        'ligand_ids': ['', ''],
        'ligand_SMILES': ['', '']
    })
    print(f"Using sample data: {len(test_sequences)} sequences")

Test sequences file not found. This notebook expects test_sequences.csv in the input directory.
Creating sample structure for development...
Using sample data: 2 sequences


## Structure Prediction Model

**TODO**: Implement your RNA 3D structure prediction model here.

This is a placeholder that generates random coordinates. Replace this with your actual model.

In [ ]:
"""
FAST RNA 3D STRUCTURE PREDICTOR
Optimized for Kaggle time constraints - completes in seconds, not minutes
Uses heuristic pairing + A-form geometry instead of slow O(n³) folding
"""

import numpy as np

def predict_rna_structure(sequence: str, prediction_number: int) -> np.ndarray:
    """Main function for Kaggle - FAST VERSION for time constraints"""
    n = len(sequence)
    coords = np.zeros((n, 3))
    
    # Parameters
    BACKBONE = 5.9
    np.random.seed(hash(sequence) % 2**32 + prediction_number * 1000)
    
    # FAST: Use simple heuristic-based pairing instead of slow folding
    # Look for GC/AU pairs with appropriate spacing
    pairs = []
    for i in range(0, n - 10):
        for j in range(i + 10, min(i + 30, n)):
            pair = sequence[i] + sequence[j]
            # Only add strong pairs (GC) or common pairs (AU)
            if pair in ['GC', 'CG', 'AU', 'UA']:
                pairs.append((i, j))
                break  # Only one pair per position
    
    # Build helices from pairs
    helices = []
    if pairs:
        sp = sorted(pairs)
        curr = [sp[0]]
        for k in range(1, len(sp)):
            if sp[k][0] == sp[k-1][0]+1 and sp[k][1] == sp[k-1][1]-1:
                curr.append(sp[k])
            else:
                if len(curr) >= 2:
                    helices.append((curr[0][0], curr[0][1], len(curr)))
                curr = [sp[k]]
        if len(curr) >= 2:
            helices.append((curr[0][0], curr[0][1], len(curr)))
    
    # Build 3D structure
    placed = set()
    pos = np.zeros(3)
    
    # Helix geometry
    RADIUS = 5.25
    RISE = 2.8
    TWIST = 32.7 * np.pi / 180
    
    for hidx, (hs, he, hl) in enumerate(helices):
        rot_base = prediction_number * np.pi / 5
        tilt = (prediction_number - 1) * np.pi / 12
        
        for k in range(hl):
            i, j = hs + k, he - k
            if i >= n or j >= n or i in placed or j in placed:
                continue
            
            ang = k * TWIST + rot_base
            z = k * RISE
            
            x1, y1 = RADIUS * np.cos(ang), RADIUS * np.sin(ang)
            coords[i] = pos + np.array([
                x1 * np.cos(tilt) - z * np.sin(tilt),
                y1,
                x1 * np.sin(tilt) + z * np.cos(tilt)
            ])
            
            x2, y2 = -RADIUS * np.cos(ang), -RADIUS * np.sin(ang)
            coords[j] = pos + np.array([
                x2 * np.cos(tilt) - z * np.sin(tilt),
                y2,
                x2 * np.sin(tilt) + z * np.cos(tilt)
            ])
            
            placed.add(i)
            placed.add(j)
        
        pos += np.array([25, 0, 0])
    
    # Fill in unpaired bases
    for i in range(n):
        if i in placed:
            continue
        
        prev = coords[i-1] if i > 0 and i-1 in placed else np.zeros(3)
        d = np.random.randn(3)
        d = d / (np.linalg.norm(d) + 1e-10)
        
        scale = 0.8 if prediction_number % 3 == 1 else 1.3 if prediction_number % 3 == 2 else 1.0
        coords[i] = prev + d * BACKBONE * scale
        placed.add(i)
    
    # Simple energy minimization (5 iterations only for speed)
    for step in range(5):
        forces = np.zeros_like(coords)
        for i in range(n-1):
            v = coords[i+1] - coords[i]
            d = np.linalg.norm(v)
            if d > 0:
                f = (d - BACKBONE) * 0.1 * v / d
                forces[i] += f
                forces[i+1] -= f
        coords += forces * 0.1
    
    # Add diversity
    temp = 0.3 + prediction_number * 0.15
    coords += np.random.normal(0, temp, coords.shape)
    
    # Random rotation
    axis = np.random.randn(3)
    axis /= (np.linalg.norm(axis) + 1e-10)
    ang = prediction_number * np.pi / 6
    
    K = np.array([[0, -axis[2], axis[1]], 
                 [axis[2], 0, -axis[0]], 
                 [-axis[1], axis[0], 0]])
    R = np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * (K @ K)
    coords = coords @ R.T
    
    # Center
    coords -= coords.mean(axis=0)
    return coords

Testing improved prediction function:
Sequence length: 9

Prediction 1:
  Shape: (9, 3)
  First residue (should NOT be zero): [ 11.08092009  -3.436784   -15.5315039 ]
  Coordinate range: x=[2.32, 11.08], y=[-3.44, 40.51], z=[-15.53, -0.69]
  No zeros: True

Prediction 2:
  Shape: (9, 3)
  First residue (should NOT be zero): [ -3.07223026 -15.42980255  -7.80532077]
  Coordinate range: x=[-9.15, 6.24], y=[-20.22, 1.32], z=[-7.81, 13.82]
  No zeros: True

Prediction 3:
  Shape: (9, 3)
  First residue (should NOT be zero): [-13.7006483   -7.69399592 -23.54174489]
  Coordinate range: x=[-19.51, -4.23], y=[-20.28, -6.52], z=[-27.71, -8.94]
  No zeros: True

Prediction 4:
  Shape: (9, 3)
  First residue (should NOT be zero): [-10.82478515   5.47578713 -11.5800331 ]
  Coordinate range: x=[-13.55, -7.97], y=[1.98, 6.66], z=[-32.92, -10.64]
  No zeros: True

Prediction 5:
  Shape: (9, 3)
  First residue (should NOT be zero): [ 15.08517618 -18.96005876 -11.32383199]
  Coordinate range: x=[15.09, 

# Quick test with timing
import time

test_sequences_list = [
    ("Short", "GGCGUAGUCC"),  # 10nt
    ("Medium", "GGCGUAGUCC" * 5),  # 50nt
    ("Long", "GGCGUAGUCC" * 20),  # 200nt
    ("VeryLong", "GGCGUAGUCC" * 50),  # 500nt
]

print("Testing prediction speed:")
for name, seq in test_sequences_list:
    start = time.time()
    try:
        coords = predict_rna_structure(seq, 1)
        elapsed = time.time() - start
        print(f"  {name} ({len(seq)}nt): {elapsed:.2f}s - Shape: {coords.shape}")
    except Exception as e:
        print(f"  {name} ({len(seq)}nt): ERROR - {e}")

print("\n✓ Speed test complete!")

In [18]:
# Generate submission template
submission_df = generate_submission_template(test_sequences)

print(f"Submission template created with {len(submission_df)} rows")
print(f"Number of unique targets: {test_sequences['target_id'].nunique()}")

# Generate predictions for all sequences
print("\nGenerating predictions...")
print(f"Total sequences to process: {len(test_sequences)}")

import time
start_time = time.time()

for idx, row in test_sequences.iterrows():
    target_id = row['target_id']
    sequence = row['sequence']
    
    # Progress for EVERY sequence
    print(f"[{idx+1}/{len(test_sequences)}] Processing {target_id} (length: {len(sequence)})", flush=True)
    
    # Optional: Load MSA if available (for advanced models)
    # msa_data = read_msa_file(target_id)
    # if msa_data:
    #     print(f"  Loaded MSA for {target_id}: {len(msa_data)} sequences")
    
    # Generate 5 predictions per sequence
    for pred_num in range(1, 6):
        pred_start = time.time()
        coords = predict_rna_structure(sequence, pred_num)
        pred_time = time.time() - pred_start
        
        if pred_time > 5:  # Warn if slow
            print(f"  Prediction {pred_num} took {pred_time:.1f}s", flush=True)
        
        # Update submission DataFrame with coordinates
        # ID format: target_id_resid
        # More efficient: create all IDs at once and update in batch
        submission_ids = [f"{target_id}_{resid}" for resid in range(1, len(sequence) + 1)]
        mask = submission_df['ID'].isin(submission_ids)
        
        # Map coordinates to the correct rows
        for i, submission_id in enumerate(submission_ids):
            row_mask = (submission_df['ID'] == submission_id) & mask
            if row_mask.sum() > 0:
                submission_df.loc[row_mask, f'x_{pred_num}'] = coords[i, 0]
                submission_df.loc[row_mask, f'y_{pred_num}'] = coords[i, 1]
                submission_df.loc[row_mask, f'z_{pred_num}'] = coords[i, 2]
    
    elapsed = time.time() - start_time
    avg_time = elapsed / (idx + 1)
    remaining = avg_time * (len(test_sequences) - idx - 1)
    print(f"  ✓ Done. Elapsed: {elapsed:.1f}s, Est. remaining: {remaining/60:.1f}min", flush=True)

print("\nAll predictions generated!")

Submission template created with 20 rows
Number of unique targets: 2

Generating predictions...

All predictions generated!


## Validate and Save Submission

In [ ]:
# Validate submission format
try:
    validate_submission(submission_df, test_sequences)
except Exception as e:
    print(f"Validation error: {e}")
    raise

# Display sample of submission
print("\nSample submission:")
print(submission_df.head(10))

# ============================================================
# COMPETITION SUBMISSION - Save to submission.csv
# Kaggle will automatically find this file in the working directory
# ============================================================

# Save submission file (REQUIRED: must be named 'submission.csv')
# save_submission clips coordinates to valid range and saves the file
OUTPUT_FILE = "submission.csv"

# Method 1: Use save_submission function (clips coordinates)
save_submission(submission_df, OUTPUT_FILE)

# Method 2: Direct save as backup (ensures file is created)
# Clip coordinates before saving
coord_cols = [col for col in submission_df.columns if col.startswith(('x_', 'y_', 'z_'))]
for col in coord_cols:
    submission_df[col] = np.clip(submission_df[col].values, -999.999, 9999.999)

# Save directly to ensure file exists
submission_df.to_csv(OUTPUT_FILE, index=False)

# Verify file was created (Kaggle requires this file to exist)
assert os.path.exists(OUTPUT_FILE), "❌ submission.csv not created!"

# Verify file is not empty
file_size = os.path.getsize(OUTPUT_FILE)
assert file_size > 0, f"❌ submission.csv is empty! Size: {file_size} bytes"

# Verify we can read it back
df_check = pd.read_csv(OUTPUT_FILE)
assert len(df_check) > 0, "❌ submission.csv has no rows!"
assert 'ID' in df_check.columns, "❌ submission.csv missing ID column!"

print("\n" + "=" * 60)
print("✅ SUCCESS: submission.csv created and ready for submission!")
print("=" * 60)
print(f"File: {OUTPUT_FILE}")
print(f"Shape: {submission_df.shape}")
print(f"Columns: {len(submission_df.columns)}")
print(f"Rows: {len(submission_df)}")
print(f"File size: {file_size} bytes ({file_size / 1024:.2f} KB)")
print(f"Full path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Working directory: {os.getcwd()}")
print("=" * 60)

# Final verification - show first few rows
print("\nFirst 3 rows of saved file:")
print(df_check.head(3))
print("\n✅ File verified and ready for submission!")

Submission validation passed!

Sample submission:
          ID resname  resid        x_1        y_1        z_1        x_2  \
0   1ABC_A_1       G      1  11.252194 -17.718521 -15.299734  -4.671788   
1   1ABC_A_2       G      2   8.929407 -17.519883  -9.729908  -4.197552   
2   1ABC_A_3       C      3   4.792035 -18.813700  -5.457572  -0.726475   
3   1ABC_A_4       G      4   1.841184 -19.193209  -0.339227   4.799047   
4   1ABC_A_5       U      5  -1.517585 -18.734958   4.533043  10.785401   
5   1ABC_A_6       A      6  -5.243404 -19.437130   9.109491  15.492899   
6   1ABC_A_7       G      7  -7.673164 -19.308144  14.603540  17.587225   
7   1ABC_A_8       U      8 -11.163559 -19.220508  19.371496  16.562381   
8   1ABC_A_9       C      9 -14.640439 -19.716293  24.126009  12.901080   
9  1ABC_A_10       C     10 -17.986990 -20.006100  28.977551   7.921732   

         y_2        z_2        x_3       y_3        z_3       x_4        y_4  \
0  -9.211746  -6.905524   6.023801 -0.783631

Submission saved to submission.csv
Shape: (20, 18)
Columns: ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1', 'x_2', 'y_2', 'z_2', 'x_3', 'y_3', 'z_3', 'x_4', 'y_4', 'z_4', 'x_5', 'y_5', 'z_5']

✓ Submission file created successfully: submission.csv
  File size: 5.88 KB
  Full path: /Users/nickmoore/kagglecomp/submission.csv


In [ ]:
# ============================================================
# FINAL VERIFICATION - Debug cell to verify submission.csv exists
# ============================================================
import glob

print("=" * 60)
print("FINAL VERIFICATION - Files in working directory:")
print("=" * 60)

# List all files
all_files = glob.glob('*')
for f in sorted(all_files):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f"  📄 {f} ({size} bytes)")
    else:
        print(f"  📁 {f}/")

print("\n" + "=" * 60)
print("Checking for submission.csv:")
print("=" * 60)

if os.path.exists('submission.csv'):
    size = os.path.getsize('submission.csv')
    print(f"✅ submission.csv EXISTS - {size} bytes")
    
    # Read and verify
    df_final = pd.read_csv('submission.csv')
    print(f"✅ File readable - Shape: {df_final.shape}")
    print(f"✅ Columns: {list(df_final.columns)[:5]}... ({len(df_final.columns)} total)")
    print(f"✅ First ID: {df_final['ID'].iloc[0]}")
    print(f"✅ Last ID: {df_final['ID'].iloc[-1]}")
    
    # Check for required columns
    required = ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']
    missing = [col for col in required if col not in df_final.columns]
    if missing:
        print(f"❌ Missing columns: {missing}")
    else:
        print("✅ All required columns present")
    
    print("\n" + "=" * 60)
    print("🎉 READY FOR SUBMISSION!")
    print("=" * 60)
else:
    print("❌ submission.csv NOT FOUND!")
    print("\nAvailable files:")
    print(os.listdir('.'))
